# Descriptive Statistics in Practice

**Topic:** Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, VBox, HTML

from IPython.display import display, clear_output
from scipy import stats

titanic = sns.load_dataset("titanic")
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this session you will be able to:

- **Interpret** mean, median, standard deviation, skewness, and kurtosis values for real data columns
- **Explain** what a gap between the mean and median reveals about a column's distribution
- **Connect** statistical summary values to decisions about how to clean and prepare that column

> **Tip:** Select `fare` from the dropdown. The gap between its mean and median is dramatic. Then select `age` and compare. Two columns, two very different statistical stories.

---
## How we got here

In the statistics working sessions we built intuition for mean, median, standard deviation, skewness, and correlation. Now we apply all of that to real data for the first time. This notebook bridges the statistics folder and the EDA folder. Specifically, this is where `statistics/04_central_tendency.ipynb`, `statistics/05_dispersion.ipynb`, and `statistics/07_correlation_covariance.ipynb` all become directly useful. From this point forward, statistical reasoning is the tool we use to understand data.

---
## Why this matters for data science

Descriptive statistics are the vocabulary of EDA. When a data scientist says "the fare column is right-skewed with high kurtosis," that carries a precise meaning: the distribution has a long right tail and many values clustered close to zero, with a few extreme values pulling the mean far from the median. That description has direct implications for modeling: the column may need a log transformation before being fed to a linear model. Statistics is not separate from data science. It is the language data science is written in.

---
## Try it yourself

In [ ]:
out = Output()

numeric_cols = titanic.select_dtypes(include="number").columns.tolist()

col_dropdown = Dropdown(
    options=numeric_cols,
    value="age",
    description="Column:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="300px"),
)

def interpret_skew(skew):
    if abs(skew) < 0.5:
        return "roughly symmetric — mean and median are close"
    elif skew > 0.5:
        return "right-skewed — a long tail of high values pulls the mean up"
    else:
        return "left-skewed — a long tail of low values pulls the mean down"

def interpret_kurtosis(kurt):
    if kurt > 1:
        return "leptokurtic — sharper peak, heavier tails than a normal distribution"
    elif kurt < -1:
        return "platykurtic — flatter peak, lighter tails than a normal distribution"
    else:
        return "mesokurtic — close to a normal distribution in shape"

def render(col):
    series = titanic[col].dropna()
    mean_val   = series.mean()
    median_val = series.median()
    std_val    = series.std()

    _not_meaningful = '<span style="color:#ADB5BD;font-style:italic">—</span>'
    _note_text      = '<em style="color:#ADB5BD">not meaningful for binary/categorical data</em>'

    is_low_cardinality = series.nunique() <= 5

    if is_low_cardinality:
        skew_value = _not_meaningful
        skew_note  = _note_text
        kurt_value = _not_meaningful
        kurt_note  = _note_text
    else:
        skew_val   = series.skew()
        kurt_val   = series.kurtosis()
        skew_value = f"{skew_val:.3f}"
        skew_note  = f"Shape of the distribution: {interpret_skew(skew_val)}."
        kurt_value = f"{kurt_val:.3f}"
        kurt_note  = f"Tail behavior: {interpret_kurtosis(kurt_val)}."

    rows = [
        ("Mean", f"{mean_val:.3f}",
         f"The average {col} across all passengers (from <em>statistics/04_central_tendency.ipynb</em>). "
         f"Sensitive to outliers."),
        ("Median", f"{median_val:.3f}",
         f"The middle value when passengers are sorted by {col}. "
         f"More robust to extreme values than the mean."),
        ("Std Dev", f"{std_val:.3f}",
         f"Typical distance from the mean (from <em>statistics/05_dispersion.ipynb</em>). "
         f"Larger std dev = more spread out."),
        ("Skewness", skew_value, skew_note),
        ("Kurtosis", kurt_value, kurt_note),
    ]

    table_rows = "".join(
        f'<tr>'
        f'<td style="padding:8px 14px;font-weight:600;background:#F1F3F5;width:110px;">{r[0]}</td>'
        f'<td style="padding:8px 14px;font-family:monospace;color:{PALETTE["primary"]};width:90px;">{r[1]}</td>'
        f'<td style="padding:8px 14px;font-size:13px;color:#495057;line-height:1.5;">{r[2]}</td>'
        f'</tr>'
        for r in rows
    )
    html = (
        f'<div style="font-family:Inter,Arial,sans-serif;max-width:700px;">'
        f'<div style="font-size:13px;color:#868E96;margin-bottom:8px;">'
        f'n = {len(series)} non-null values</div>'
        f'<table style="border-collapse:collapse;width:100%;font-size:14px;">'
        f'{table_rows}'
        f'</table></div>'
    )
    with out:
        clear_output(wait=True)
        display(HTML(html))

def on_change(change):
    render(col_dropdown.value)

col_dropdown.observe(on_change, names="value")

display(VBox([col_dropdown, out]))
render(col_dropdown.value)

---
## What's happening?

Five statistics summarize the shape of any numeric column:

| Statistic | Symbol | What it tells you | Reference |
|-----------|--------|------------------|-----------|
| Mean | μ | The average — pulled toward outliers | `statistics/04_central_tendency.ipynb` |
| Median | M | The middle value — robust to extremes | `statistics/04_central_tendency.ipynb` |
| Std Dev | σ | Typical spread around the mean | `statistics/05_dispersion.ipynb` |
| Skewness | γ₁ | Direction and degree of asymmetry | `statistics/05_dispersion.ipynb` |
| Kurtosis | γ₂ | How heavy the tails are | `statistics/05_dispersion.ipynb` |

Remember from `statistics/04_central_tendency.ipynb`? Here is where that matters: if mean > median, the column has a right skew. If mean ≈ median, the distribution is roughly symmetric. That difference changes how you handle the column downstream.

---
## Real-world example: Titanic Dataset

The chart below compares mean, median, and standard deviation across the four numeric columns in the Titanic dataset.

Notice:
- For **`fare`**, the mean (£32.2) is much higher than the median (£14.5), which tells us the distribution is strongly right-skewed with a few very expensive tickets pulling the average up
- For **`age`**, the mean (29.7) and median (28) are close, indicating a roughly symmetric distribution with mild right skew
- **`sibsp`** and **`parch`** both have very low medians (0) but non-trivial means, suggesting most passengers traveled alone with a few large family groups

> **Discussion question:** The mean age is 29.7 but the median is 28. What does that small gap tell you about the distribution of ages on the Titanic?

In [ ]:
numeric_cols = ["age", "fare", "sibsp", "parch"]
stats_data = {}
for col in numeric_cols:
    s = titanic[col].dropna()
    stats_data[col] = {
        "mean":   s.mean(),
        "median": s.median(),
        "std":    s.std(),
    }

fig = go.Figure()
x = numeric_cols

fig.add_trace(go.Bar(
    name="Mean",
    x=x,
    y=[stats_data[c]["mean"] for c in x],
    marker_color=PALETTE["primary"],
))
fig.add_trace(go.Bar(
    name="Median",
    x=x,
    y=[stats_data[c]["median"] for c in x],
    marker_color=PALETTE["accent"],
))
fig.add_trace(go.Bar(
    name="Std Dev",
    x=x,
    y=[stats_data[c]["std"] for c in x],
    marker_color=PALETTE["secondary"],
    opacity=0.7,
))

layout = base_layout(
    title="Mean, Median, and Std Dev for Titanic Numeric Columns",
    xaxis_title="Column",
    yaxis_title="Value",
)
layout.update(barmode="group")
fig.update_layout(layout)
fig.show()

### How descriptive statistics connect to modeling decisions

| Statistical finding | What it implies for modeling |
|---------------------|------------------------------|
| Mean far from median (right skew) | Consider log-transforming before feeding to a linear model |
| Very high std dev relative to mean | Feature may need standardization (Z-score scaling) |
| Skewness > 1 or < -1 | Distribution departs significantly from normal |
| Kurtosis > 3 | Heavy tails — outliers will have outsized influence |
| Mean ≈ median, low skewness | Column is well-behaved — less preprocessing needed |

---
## Key takeaway

> **Descriptive statistics are not just numbers to report. They are a diagnostic tool that tells you exactly how to prepare each column before it enters a model.**

---
*Next up: 07b_groupby_and_pivot_tables.ipynb — summarizing groups with groupby, pivot tables, and crosstab*